# 01 — News Data Collection (Web Scraping)

**Name:** …
**NIM:** …
**Video explanation:** *(paste the public link here)*

DTSC6008001 Text Mining — Final Project, Task 1 (5%).

**Goal:** collect at least 5,000 Indonesian news articles from 5 news domains (the class labels), and document the data source, the scraping method and tools, the collection period and the number of records.

## 1. Data source

**TODO — write this section in your own words.** Points to cover:

- Which site and which 5 channels, and why. Each channel becomes one label, so the labels come from the publisher's own categories.
- The robots.txt check (done on 10 Sep 2026; re-check before your full run):
  - `detik.com` and every channel subdomain: `User-agent: *` → `Allow: /`.
  - `kompas.com`: prohibits scraping and text/data mining without written permission, so it is not used.
- All 5 labels come from the same site, so the classifier cannot learn *which site* instead of *which topic*.

## 2. How Detik is organised

Notes from inspecting the site on 10 Sep 2026. **Rewrite them in your own words.** You need to be able to explain this in the video.

| What | Where |
|---|---|
| A channel's daily index | `https://<channel>/indeks?date=YYYY-MM-DD&page=N`: 20 articles per page; page 1 links to the day's last page |
| Article URL | `https://<channel>/<section>/d-<id>/<slug>`: `<id>` is unique (used to remove duplicates); `<section>` shows which pages are photo/video galleries to skip |
| Article text | every `<p>` inside `div.detail__body-text` (articles split over several pages are already complete in the HTML) |
| Title, date, author, tags | `<meta>` tags `og:title`, `dtk:publishdate`, `dtk:author`, `dtk:keywords`: the same on both of Detik's page designs (detikInet uses a newer design) |

**Quirk:** the index sometimes returns a different page of the same day than the one requested (asked for page 1, got page 10). A correct page always contains articles not seen yet, so `collect_urls` asks again when a page brings nothing new.

To see this yourself: open an index page in Chrome → F12 → right-click an article title → *Inspect*.

In [ ]:
import random
import re
import time
from datetime import date, timedelta
from pathlib import Path

import pandas as pd
import requests
from bs4 import BeautifulSoup
from tqdm.auto import tqdm

d:\text-mining-aol\text-mining\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 3. Scraping functions

- `fetch(url)` downloads a page and waits 1.5 s afterwards, so the scraper never sends more than one request every 1.5 seconds.
- `collect_urls(base, day)` returns every article URL a channel listed on one day, skipping photo, video and infographic pages.
- `parse_article(url)` downloads one article and returns its fields. `content` is kept **raw** here; cleaning happens in `02_preprocessing.ipynb`, so the cleaning rules can change without scraping again.

In [2]:
HEADERS = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                         "(KHTML, like Gecko) Chrome/128.0 Safari/537.36"}
SKIP_SECTIONS = re.compile(r"foto|video|detiktv|grafis", re.I)  # photo/video/infographic pages have little text


def fetch(url, params=None, delay=1.5):
    """GET a page politely (pause after every request) and return it parsed."""
    r = requests.get(url, params=params, headers=HEADERS, timeout=20)
    time.sleep(delay)
    r.raise_for_status()
    return BeautifulSoup(r.text, "lxml")


def collect_urls(base, day, max_tries=4):
    """All article URLs a Detik channel listed on one day, e.g. base='https://finance.detik.com'."""
    pattern = re.compile(re.escape(base) + r"/([\w-]+)/d-(\d+)/[\w-]+")
    seen, urls = set(), {}
    page = last_page = 1
    while page <= last_page:
        for _ in range(max_tries):
            soup = fetch(f"{base}/indeks", params={"date": day.isoformat(), "page": page})
            links = [a["href"] for a in soup.select("a[href]")]
            ids = set(re.findall(r"/d-(\d+)/", " ".join(links)))
            if ids - seen:  # Detik sometimes returns a page we already have; if so, ask again
                break
        seen |= ids
        if page == 1:  # page 1 links to the day's last page ("1 2 3 4 5 ... 15")
            last_page = max([int(n) for h in links for n in re.findall(r"[?&]page=(\d+)", h)], default=1)
        for h in links:
            m = pattern.match(h)
            if m and not SKIP_SECTIONS.search(m.group(1)):
                urls[m.group(2)] = m.group(0)  # key = article id, so the same article isn't kept twice
        page += 1
    return list(urls.values())


def meta(soup, **attrs):
    tag = soup.find("meta", attrs=attrs)
    return tag.get("content") if tag else None


def parse_article(url):
    """Fetch one Detik article and return its fields (content = raw paragraphs, cleaned in 02_preprocessing)."""
    soup = fetch(url)
    body = soup.select_one("div.detail__body-text")
    paragraphs = [p.get_text(" ", strip=True) for p in body.find_all("p")] if body else []
    return {
        "url": url,
        "title": meta(soup, property="og:title"),
        "date": meta(soup, name="dtk:publishdate"),
        "author": meta(soup, name="dtk:author"),
        "tags": meta(soup, name="dtk:keywords"),
        "content": "\n".join(p for p in paragraphs if p),
    }

## 4. Configuration

**TODO — decide these, and justify them in section 1:**

- **Labels:** pick 5 keys from `CHANNELS`.
  - Very different channels make classification easy (both models near 95%). One close pair (e.g. ekonomi vs teknologi) makes it harder, and the model comparison more interesting.
  - `nasional` covers almost everything, and `otomotif` includes MotoGP, which overlaps `olahraga`.
- **Period and quota:** kesehatan is the smallest channel (~18 usable articles a day). 3 months at 15 articles per label per day gives each label about 1,300 articles, a buffer above 5,000 in total after cleaning.

In [3]:
# Usable articles per weekday, counted on 3 Jun and 12 Aug 2026 (photo/video pages skipped)
CHANNELS = {
    "nasional":   ["https://news.detik.com"],                                        # ~200/day
    "ekonomi":    ["https://finance.detik.com"],                                     # ~50/day
    "teknologi":  ["https://inet.detik.com"],                                        # ~30/day
    "olahraga":   ["https://sport.detik.com", "https://sport.detik.com/sepakbola"],  # ~65/day
    "kesehatan":  ["https://health.detik.com"],                                      # ~18/day
    "hiburan":    ["https://hot.detik.com"],                                         # ~22/day
    "otomotif":   ["https://oto.detik.com"],                                         # ~22/day
    "pendidikan": ["https://www.detik.com/edu"],                                     # ~32/day
}

LABELS = []                  # TODO: your 5 keys from CHANNELS
START = date(2026, 6, 1)     # suggested period
END = date(2026, 8, 31)
PER_DAY = 15                 # articles per label per day
SEED = 42

RAW_DIR = Path("../data/raw")
RAW_DIR.mkdir(parents=True, exist_ok=True)
URLS_CSV = RAW_DIR / "urls.csv"
ARTICLES_CSV = RAW_DIR / "articles.csv"

## 5. Pilot test

Run the functions on one channel and one day. Then open the same articles in your browser and check that the title, date and full text match.

In [4]:
test_urls = collect_urls("https://health.detik.com", date(2026, 6, 3))
print(len(test_urls), "article URLs")

pilot = pd.DataFrame([parse_article(u) for u in test_urls[:3]])
pilot["words"] = pilot["content"].str.split().str.len()
pilot[["date", "title", "words", "url"]]

FeatureNotFound: Couldn't find a tree builder with the features you requested: lxml. Do you need to install a parser library?

In [ ]:
# compare this with the article in your browser
print(pilot.loc[0, "url"], "\n")
print(pilot.loc[0, "content"])

## 6. Main scraping loop

**TODO — write this part yourself.** The brief doesn't allow AI to generate the complete scraping solution, and this is the part you walk through in the video. Once it's written, ask for a review or debugging help.

**Stage A: URL list → `data/raw/urls.csv`**
1. Loop over every date from `START` to `END` (step `timedelta(days=1)`) and every label in `LABELS`.
2. Call `collect_urls` for each base URL of the label (`olahraga` has two) and combine the lists.
3. Pick `PER_DAY` URLs at random: call `random.seed(SEED)` once, then `random.sample`. The index lists newest first, so taking the first ones would give only evening news.
4. Append `label, date, url` rows to `urls.csv` after every day, so a crash doesn't lose progress.

**Stage B: articles → `data/raw/articles.csv`**
1. Read `urls.csv`. If `articles.csv` already exists, skip the URLs that are in it. This is what makes a rerun continue where it stopped.
2. For each remaining URL, call `parse_article` inside `try/except`, add the label, and append the row to `articles.csv` immediately (`to_csv(mode="a", header=not ARTICLES_CSV.exists(), index=False)`).
3. Keep a list of failed URLs and their errors instead of stopping; retry them at the end.
4. Show progress with `tqdm`.

Test on 1 label × 2 days first. The full run takes roughly 4–6 hours at 1.5 s per request, so run it overnight with sleep mode turned off.

In [ ]:
# Stage A: build data/raw/urls.csv
# TODO: write this yourself (see the steps above)


In [ ]:
# Stage B: download the articles into data/raw/articles.csv
# TODO: write this yourself (see the steps above)


## 7. Checks after scraping

**TODO:**
- Number of articles per label (target ≥ 1,000 each after cleaning, ≥ 5,000 in total)
- Empty or very short `content` (some Detik articles really are only ~20 words)
- Duplicate URLs, and duplicate titles or texts
- Publish dates fall inside `START`–`END`
- Open 10 random URLs and compare them with the saved text

Keep the text raw here. `02_preprocessing.ipynb` removes what testing found inside the paragraphs:
- "SCROLL TO CONTINUE WITH CONTENT" and "ADVERTISEMENT"
- "detikcom" / "detikHealth" mentions, which give away the label
- "Jakarta -" at the start of articles
- detikevent/detikcourse promos

In [ ]:
# TODO: checks after scraping


## 8. Data collection summary

**TODO — fill in after the run** (the rubric asks for all of these):

| Item | Value |
|---|---|
| Source | detik.com, channels: … |
| Scraping method and tools | … |
| Collection period (publish dates) | … to … |
| Date scraped | … |
| Records collected | … in total (per label: …) |
| Records after cleaning | … (see `02_preprocessing.ipynb`) |
| Fields | `label`, `url`, `title`, `date`, `author`, `tags`, `content` |